# Бейзлайны для прогнозирования временных рядов

Этот notebook содержит код из главы книги "Нейросети для прогнозирования временных рядов".

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/privettoha/neural-forecast-book/blob/main/notebooks/03_baselines.ipynb)

## Установка зависимостей

In [ ]:
!pip install -q statsforecast pandas numpy matplotlib

## Подготовка данных

In [ ]:
import pandas as pd
import numpy as np

# Создаём синтетические данные для демонстрации
np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=365, freq='D')
y = 100 + np.cumsum(np.random.randn(365)) + 20 * np.sin(np.arange(365) / 7 * 2 * np.pi)

df = pd.DataFrame({
    'unique_id': 'series_1',
    'ds': dates,
    'y': y
})
print(df.head())

## Бейзлайны с использованием statsforecast

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, RandomWalkWithDrift

# Инициализируем бейзлайны
# season_length=7 для дневных данных с недельной сезонностью
models = [
    Naive(),
    SeasonalNaive(season_length=7),
    RandomWalkWithDrift()
]

sf = StatsForecast(
    models=models,
    freq='D',  # дневная гранулярность
    n_jobs=-1  # параллелизация на все ядра
)

# df должен содержать колонки: unique_id, ds, y
# unique_id — идентификатор ряда
# ds — дата
# y — значение

sf.fit(df)
forecasts = sf.predict(h=16)  # прогноз на 16 дней

# forecasts будет содержать колонки:
# unique_id, ds, Naive, SeasonalNaive, RandomWalkWithDrift
print(forecasts)

## Визуализация прогнозов

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))

# Последние 50 точек истории
history = df.tail(50)
ax.plot(history['ds'], history['y'], label='История', color='blue')

# Прогнозы
ax.plot(forecasts['ds'], forecasts['Naive'], label='Naive', linestyle='--')
ax.plot(forecasts['ds'], forecasts['SeasonalNaive'], label='SeasonalNaive', linestyle='--')
ax.plot(forecasts['ds'], forecasts['RandomWalkWithDrift'], label='Drift', linestyle='--')

ax.set_title('Сравнение бейзлайнов')
ax.set_xlabel('Дата')
ax.set_ylabel('Значение')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Бейзлайны для валютных курсов (без сезонности)

In [ ]:
# Для валютных курсов — без выраженной сезонности
models_fx = [
    Naive(),
    SeasonalNaive(season_length=5),  # рабочая неделя
    RandomWalkWithDrift()
]

sf_fx = StatsForecast(
    models=models_fx,
    freq='D',
    n_jobs=-1
)

sf_fx.fit(df)
forecasts_fx = sf_fx.predict(h=16)
print(forecasts_fx)